# fractional-stride-zero-insertion — ex1: build the zero-inserted intermediate for stride-2 ConvT

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `fractional-stride-zero-insertion`. Running the final beacon cell reports progress against the `CNN: ConvT fractional-stride zero insertion` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT fractional-stride zero insertion` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`fractional-stride-zero-insertion`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "fractional-stride-zero-insertion"
DD_SUBTOPIC = "CNN: ConvT fractional-stride zero insertion"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose2d fractional stride (zero insertion) — quick refresher

`nn.ConvTranspose2d` with `stride=S` doesn't physically stride the kernel — it **dilates the input** by inserting `S - 1` rows/cols of zeros *between every pair of adjacent input pixels*, then does a regular stride-1 convolution.

Example with `S = 2` on a 1-D input of length 4:

```
input          : [a, b, c, d]
after dilation : [a, 0, b, 0, c, 0, d]      # length = (4-1)*2 + 1 = 7
```

**Why call it 'fractional stride'.** The output advances by ONE pixel for every `1/S` input pixels — equivalent to a forward conv with stride `1/S`. That's where the name comes from.

**Shape formula** (stride-S, no padding, no output_padding):

```
H_out = (H_in - 1) * S + K
```

Compare to a forward stride-S conv `H_out = (H_in - K) // S + 1` — they are *adjoint*: ConvT(stride=S, K) reverses the spatial shape change of Conv(stride=S, K).

**Why upsampling networks use it.** GANs, U-Nets, autoencoder decoders, and stable-diffusion's VAE all use stride-2 ConvT layers to double spatial resolution at each decoder block. The zero-insertion is the mechanism that makes the output bigger than the input.

**Visualization in this drill.** You'll build the zero-inserted intermediate explicitly and imshow it next to the convolved output, so you can see the dilation rather than read about it.

### Exercise 1 — build the zero-inserted intermediate for stride-2 ConvT

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze stride-`S` ConvTranspose2d by explicitly building the `(S-1)`-zero-dilated intermediate input and confirming a stride-1 conv on that intermediate reproduces `F.conv_transpose2d(stride=S)`.
> Keywords: ConvTranspose2d, fractional-stride, zero-insertion, upsample, visualization
> ```

**KCs targeted:** `convT-stride-zero-dilation`, `convT-stride-shape-formula`

Implement `ex1_zero_insert(x, s)`. Given input `x: (B, C, H, W)` and an integer upsample stride `s >= 1`, return the **zero-inserted** tensor of shape `(B, C, (H-1)*s + 1, (W-1)*s + 1)` where:

- Original pixel `x[b, c, i, j]` lives at position `[b, c, i*s, j*s]` in the output.
- Every other position (between original pixels) is **0**.

Example for `s = 2`, 1-D input `[a, b, c, d]` (length 4):

```
input   : [a, b, c, d]
output  : [a, 0, b, 0, c, 0, d]      # length (4-1)*2 + 1 = 7
```

**Approach.**
1. Compute `H_out = (H - 1) * s + 1` and `W_out = (W - 1) * s + 1`.
2. Allocate `y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)`.
3. Scatter `x` into `y` with step `s`: `y[:, :, ::s, ::s] = x`.

**Edge case.** `s = 1` → output equals input (no dilation, no zeros inserted).

After your implementation, the test runs a stride-1 conv on your zero-inserted output and compares to the *real* `F.conv_transpose2d(x, weight, stride=s)`. The two must match to fp tolerance — this is the equivalence the atom teaches.

The visualization renders one 8×8 input feature map next to its zero-inserted 15×15 dilation so you can SEE the (s-1) zero rows/cols between every pair of original rows/cols.

In [ ]:
def ex1_zero_insert(x: Tensor, s: int) -> Tensor:
    B, C, H, W = x.shape
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y


<details><summary>Solution</summary>

```python
def ex1_zero_insert(x: Tensor, s: int) -> Tensor:
    B, C, H, W = x.shape
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y
```

**Why the shape formula is `(H-1)*s + 1`.** There are `H` input pixels and `H - 1` *gaps* between adjacent pixels. Each gap gets filled with `s - 1` zeros — that's `(H-1)*(s-1)` inserted zeros. Total length = `H + (H-1)*(s-1) = (H-1)*s + 1`. The same logic gives the width.

**Why `y[..., ::s, ::s] = x` works.** Python slice `::s` is 'every s-th index starting from 0'. For `s=2` on a length-7 axis, that's indices `[0, 2, 4, 6]` — exactly 4 positions, matching the 4 original pixels of a length-4 input. The remaining positions stay zero (we allocated with `t.zeros`).

**Why this is the meaning of 'fractional stride'.** The downstream stride-1 conv now slides over a 2x-bigger input — so the OUTPUT advances at half the rate of the original input. That's the 'stride 1/2' interpretation: one output pixel per ½ input pixel.

**Equivalence in 3 lines.** Compose this drill with the `convT-as-flipped-padded-conv` drill from batch-3 and you've built `F.conv_transpose2d` for any stride from scratch:
```
x_di     = zero_insert(x, s)
x_padded = F.pad(x_di, (K-1,) * 4)
w_eq     = w.flip(-1).flip(-2).transpose(0, 1)
y        = F.conv2d(x_padded, w_eq)        # == F.conv_transpose2d(x, w, stride=s)
```
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()